# Datathon 2026 - The Gridbreaker
Notebook này được thiết lập để giải các bài tập trong `ex1.md`.

In [55]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập hiển thị
pd.set_option('display.max_columns', None)


print("Libraries imported successfully!")

Libraries imported successfully!


Q1

In [56]:
#Load Orders
orders = pd.read_csv('data/orders.csv')

#Process date format
orders['order_date']= pd.to_datetime(orders['order_date'])

#Sort base on customer and date
orders= orders.sort_values(by = ['customer_id','order_date'])

#Distance between customer buy time
orders['days_diff'] = orders.groupby('customer_id')['order_date'].diff().dt.days

#Remove NaN
inter_order_gap = orders['days_diff'].dropna()

#Calculate median
median_gap = inter_order_gap.median()

print(f"Giá trị trung vị số ngày giữa hai lần mua: {median_gap} ngày")

Giá trị trung vị số ngày giữa hai lần mua: 144.0 ngày


Q2

In [57]:
#Load Products
products = pd.read_csv('data/products.csv')
# Tính Tỷ suất lợi nhuận gộp (Gross Margin) cho từng sản phẩm
products['gross_margin']=(products['price'] - products['cogs'])/ products['price']

#Group by Segment column and calculate Mean
segment_margin = products.groupby('segment')['gross_margin'].mean()

#Tìm segment có giá trị trung bình cao nhất
highest_segment = segment_margin.idxmax()

print(f"Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất: {highest_segment}")

Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất: Standard


Q3

In [58]:
# Load returns
returns = pd.read_csv('data/returns.csv')

# returns.csv có sẵn product_id, join trực tiếp với products để lấy category
returns_with_category = returns.merge(products[['product_id', 'category']], on='product_id', how='left')

# Lọc chỉ Streetwear
streetwear_returns = returns_with_category[returns_with_category['category'] == 'Streetwear']

# Đếm lý do trả hàng
reason_counts = streetwear_returns['return_reason'].value_counts()

print("Lý do trả hàng trong danh mục Streetwear:")
print(reason_counts)
print(f"\nLý do xuất hiện nhiều nhất: {reason_counts.idxmax()}")

Lý do trả hàng trong danh mục Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Lý do xuất hiện nhiều nhất: wrong_size


Q4

In [59]:
# Load web_traffic
web_traffic = pd.read_csv('data/web_traffic.csv')

# Tính bounce_rate trung bình theo traffic_source
avg_bounce = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()

print("Tỷ lệ thoát trung bình theo nguồn truy cập:")
print(avg_bounce)
print(f"\nNguồn có bounce_rate thấp nhất: {avg_bounce.idxmin()}")

Tỷ lệ thoát trung bình theo nguồn truy cập:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Nguồn có bounce_rate thấp nhất: email_campaign


Q5

In [60]:
# Tổng số dòng trong order_items
total_rows = len(order_items)

# Số dòng có promo_id không null
promo_rows = order_items['promo_id'].notna().sum()

# Tính tỷ lệ phần trăm
promo_pct = promo_rows / total_rows * 100

print(f"Tổng số dòng order_items: {total_rows:,}")
print(f"Số dòng có promo_id: {promo_rows:,}")
print(f"Tỷ lệ có khuyến mãi: {promo_pct:.2f}%")

Tổng số dòng order_items: 714,669
Số dòng có promo_id: 276,316
Tỷ lệ có khuyến mãi: 38.66%


Q6

In [61]:
# Load customers
customers = pd.read_csv('data/customers.csv')

# Lọc khách hàng có age_group khác null
customers_with_age = customers[customers['age_group'].notna()]

# Đếm số đơn hàng mỗi khách hàng
order_counts = orders.groupby('customer_id').size().reset_index(name='order_count')

# Merge với customers
customers_orders = customers_with_age.merge(order_counts, on='customer_id', how='left')
customers_orders['order_count'] = customers_orders['order_count'].fillna(0)

# Tính số đơn trung bình theo age_group
avg_orders_by_age = customers_orders.groupby('age_group')['order_count'].mean().sort_values(ascending=False)

print("Số đơn hàng trung bình theo nhóm tuổi:")
print(avg_orders_by_age)
print(f"\nNhóm tuổi có số đơn trung bình cao nhất: {avg_orders_by_age.idxmax()}")

Số đơn hàng trung bình theo nhóm tuổi:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: order_count, dtype: float64

Nhóm tuổi có số đơn trung bình cao nhất: 55+


Q7

In [62]:
# Load Geography
orders = pd.read_csv('data/orders.csv')
order_items = pd.read_csv('data/order_items.csv')
geography = pd.read_csv('data/geography.csv')


# 1. Chuyển đổi ngày và lọc dữ liệu Train (đến hết 31/12/2022)
orders['order_date'] = pd.to_datetime(orders['order_date'])
train_orders = orders[orders['order_date'] <= '2022-12-31'].copy()
# 2. Join orders trực tiếp với geography qua cột 'zip'
# Lưu ý: Dùng zip trong đơn hàng để tính doanh thu theo vùng chính xác nhất
orders_geo = train_orders.merge(geography[['zip', 'region']], on='zip', how='left')
# 3. Tính revenue cho từng item
order_items['revenue'] = order_items['quantity'] * order_items['unit_price'] - order_items['discount_amount']
# 4. Join order_items với thông tin vùng từ orders_geo
# Sử dụng how='inner' để đảm bảo chỉ tính các đơn hàng thuộc tập Train
final_df = order_items.merge(orders_geo[['order_id', 'region']], on='order_id', how='inner')
# 5. Tổng hợp doanh thu
revenue_by_region = final_df.groupby('region')['revenue'].sum().sort_values(ascending=False)
print("Tổng doanh thu theo từng vùng (Giai đoạn Train):")
print(revenue_by_region)
print(f"\n=> Vùng có doanh thu cao nhất: {revenue_by_region.idxmax()}")

C:\Users\nguye\AppData\Local\Temp\ipykernel_27308\3332854787.py:3: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('data/order_items.csv')


Tổng doanh thu theo từng vùng (Giai đoạn Train):
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: revenue, dtype: float64

=> Vùng có doanh thu cao nhất: East


Q8

In [63]:
# Load payments
payments = pd.read_csv('data/payments.csv')

# Lọc đơn hàng bị hủy
cancelled_orders = orders[orders['order_status'] == 'cancelled'][['order_id']]

# Join với payments để lấy payment_method
cancelled_payments = cancelled_orders.merge(payments[['order_id', 'payment_method']], on='order_id', how='left')

# Đếm phương thức thanh toán
payment_counts = cancelled_payments['payment_method'].value_counts()

print("Phương thức thanh toán trong các đơn hủy:")
print(payment_counts)
print(f"\nPhương thức được dùng nhiều nhất: {payment_counts.idxmax()}")

Phương thức thanh toán trong các đơn hủy:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

Phương thức được dùng nhiều nhất: credit_card


Q9

In [64]:
# Join order_items với products để lấy size
items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')

# Lọc chỉ 4 kích thước S, M, L, XL
sizes = ['S', 'M', 'L', 'XL']
items_filtered = items_with_size[items_with_size['size'].isin(sizes)]

# Đếm số dòng order_items theo size
items_by_size = items_filtered.groupby('size').size().rename('total_items')

# returns.csv có sẵn product_id, join với products để lấy size
returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
returns_filtered = returns_with_size[returns_with_size['size'].isin(sizes)]

# Đếm số returns theo size
returns_by_size = returns_filtered.groupby('size').size().rename('total_returns')

# Tính tỷ lệ trả hàng
return_rate = (returns_by_size / items_by_size * 100).sort_values(ascending=False)

print("Tỷ lệ trả hàng theo kích thước (%):")
print(return_rate)
print(f"\nKích thước có tỷ lệ trả hàng cao nhất: {return_rate.idxmax()}")

Tỷ lệ trả hàng theo kích thước (%):
size
S     5.651527
L     5.624978
M     5.566010
XL    5.520010
dtype: float64

Kích thước có tỷ lệ trả hàng cao nhất: S


Q10

In [65]:
# Load payments
payments = pd.read_csv('data/payments.csv')

# Tính giá trị thanh toán trung bình theo số kỳ trả góp
# Cột đúng: installments (không phải installment_plan), payment_value (không phải payment_amount)
avg_payment_by_installment = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

print("Giá trị thanh toán trung bình theo kỳ trả góp:")
print(avg_payment_by_installment)
print(f"\nKỳ trả góp có giá trị trung bình cao nhất: {avg_payment_by_installment.idxmax()} kỳ")

Giá trị thanh toán trung bình theo kỳ trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

Kỳ trả góp có giá trị trung bình cao nhất: 6 kỳ
